# RankLab: KuaiRand-Pure baseline experiment on Kaggle

Enable a **GPU accelerator** in Kaggle settings. Add a Kaggle dataset containing the official `KuaiRand-Pure/data/` directory, or set `DOWNLOAD_IF_MISSING = True` and enable Internet. Raw data and experiment outputs are not committed to GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/kushc2004/rank-lab.git'
WORKDIR = Path('/kaggle/working/rank-lab')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print(WORKDIR)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running BPR-MF.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Preferred: attach a Kaggle dataset whose root contains KuaiRand-Pure/data.
# Optional: download only from the official Zenodo source when Kaggle Internet is enabled.
DOWNLOAD_IF_MISSING = False
expected = ('log_random_4_22_to_5_08_pure.csv', 'log_standard_4_08_to_4_21_pure.csv', 'log_standard_4_22_to_5_08_pure.csv')
candidates = list(Path('/kaggle/input').glob('*/KuaiRand-Pure/data'))
if candidates:
    RAW_DIR = candidates[0]
elif DOWNLOAD_IF_MISSING:
    subprocess.run(['bash', 'scripts/download_kuairand_pure.sh'], check=True)
    RAW_DIR = WORKDIR / 'data/raw/KuaiRand-Pure/data'
else:
    raise FileNotFoundError('Attach the official KuaiRand-Pure Kaggle dataset, or enable Internet and set DOWNLOAD_IF_MISSING=True.')
missing = [name for name in expected if not (RAW_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Not an official KuaiRand-Pure data directory: missing {missing}')
print('Using raw data:', RAW_DIR)

In [ ]:
# Commands are intentionally run one at a time, in dependency order.
commands = [
    ['python', 'scripts/audit_kuairand.py'],
    ['python', 'scripts/build_features.py'],
    ['python', 'scripts/train_popularity.py'],
    ['python', 'scripts/train_bpr.py', 'device=cuda'],
    ['python', 'scripts/evaluate_exposure_gap.py'],
    ['pytest'],
]
for command in commands:
    arguments = [*command, f'raw_dir={RAW_DIR}'] if command[1].startswith('scripts/') else command
    print('\n$', ' '.join(map(str, arguments)))
    subprocess.run(arguments, check=True)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((WORKDIR / 'outputs/reports/initial_exposure_gap.md').read_text()))